In [9]:
import numpy as np
from moabb.datasets import BNCI2015_001
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mne.decoding import CSP
from moabb.evaluations import CrossSubjectEvaluation
from sklearn.pipeline import make_pipeline
from scipy import signal
from scipy.io import loadmat
import os
import mne

In [2]:

active_all_event_ids = {'769': 769, '770': 770, '771': 771, '772': 772}
active_lr_event_ids = {'769': 769, '770': 770}
unknown_event_id = {'783': 783}

In [3]:

# Define a causal bandpass filter function using a Butterworth design.
def causal_bandpass_filter(data, lowcut=8, highcut=30, fs=250, order=50):
    nyq = 0.5 * fs
    # Normalize the cutoff frequencies (Matlab's fir1 expects normalized cutoff frequencies
    low = lowcut / nyq
    high = highcut / nyq
    # Design the FIR filter. Note: order+1 coefficients are returned to match Matlab's fir1 which returns n+1 taps.
    b = signal.firwin(order + 1, [low, high], window='hamming', pass_zero=False)
    # Apply the filter causally using lfilter (this introduces a constant delay).
    filtered_data = signal.lfilter(b, [1.0], data)
    return filtered_data

In [4]:
 
data_dir = '/home/vishwa/eeg_tl/Recreating papers/2015_001/A'

# Lists to hold data for all subjects
train_active_X = []         # List to hold numpy arrays with shape (n_trials, 13, n_times) per subject
train_active_y = []         # List to hold event labels per subject
train_active_metadata = []  # List to hold event metadata per subject

# Define subject IDs (S01 to S12 based on the dataset description)
subjects = [f'S{subj:02d}' for subj in range(1, 13)]

for subj in subjects:
    # Find all .mat files for this subject (e.g., S01_session1.mat, S01_session2.mat)
    subj_files = [f for f in os.listdir(data_dir) if f.startswith(subj) and f.endswith('.mat')]
    subj_epochs_list = []

    for filename in subj_files:
        full_filename = os.path.join(data_dir, filename)
        
        # Load the .mat file
        data_dict = loadmat(full_filename, squeeze_me=True)
        data = data_dict['data']
        X = data['X'][()]  # EEG signal (n_times, n_channels)
        print(type(X))  # Should be <class 'numpy.ndarray'>
        print(X.shape if isinstance(X, np.ndarray) else "Not a NumPy array")
        print(X.dtype if isinstance(X, np.ndarray) else "No dtype available")
        X = X.T
        y = data['y'][()]  # True labels (1 for right hand, 2 for both feet)
        print(f"y type: {type(y)}")
        if isinstance(y, np.ndarray):
            print(f"y shape: {y.shape}")
        else:
            print(f"y value: {y}")
        # print(f"Number of trials: {len(trial)}")
        trial = data['trial']  # Trial start positions in samples
        fs = data['fs']  # Sampling rate (512 Hz)

        # Define channel names and types
        ch_names = ['FC3', 'FCz', 'FC4', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP3', 'CPz', 'CP4']
        ch_types = ['eeg'] * 13
        info = mne.create_info(ch_names, sfreq=fs, ch_types=ch_types)

        # Create Raw object (transpose X to (n_channels, n_times) as required by MNE)
        raw = mne.io.RawArray(X, info, verbose=False)

        # Compute cue onset samples (cue starts 3 seconds after trial start)
        cue_samples = trial + int(3 * fs)

        # Create events array: [sample, 0, label]
        events = np.column_stack((cue_samples, np.zeros(len(cue_samples), dtype=int), y))

        # Define epoching parameters
        tmin = 0.5  
        tmax = 4.5   

        # Create epochs with baseline correction using the reference period
        epochs = mne.Epochs(
            raw,
            events,
            event_id={'1': 1, '2': 2},  # 1: right hand, 2: both feet
            tmin=tmin,
            tmax=tmax,
            baseline=None,  # Baseline from t=-3 to t=0
            preload=True,
            verbose=False
        )
        subj_epochs_list.append(epochs)

    # Concatenate epochs across sessions for this subject
    if len(subj_epochs_list) > 1:
        subj_epochs = mne.concatenate_epochs(subj_epochs_list)
    else:
        subj_epochs = subj_epochs_list[0]

    # Get the epoch data
    subj_data = subj_epochs.get_data()

    # Apply causal bandpass filter to each trial and channel
    n_trials, n_channels, n_times = subj_data.shape
    subj_filtered_data = np.empty_like(subj_data)
    for trial in range(n_trials):
        for ch in range(n_channels):
            subj_filtered_data[trial, ch, :] = causal_bandpass_filter(
                subj_data[trial, ch, :],
                lowcut=8,   # Lower bound of sensorimotor rhythm
                highcut=30, # Upper bound of sensorimotor rhythm
                fs=fs,
                order=50    # Filter order
            )

    # Append processed data, labels, and metadata
    train_active_X.append(subj_filtered_data)
    train_active_y.append(subj_epochs.events[:, 2])  # Labels are in the third column
    train_active_metadata.append(subj_epochs.events)

    # Print shape to verify
    print(f"Subject {subj}: Epoch data shape {subj_filtered_data.shape}")

print("Loaded data for", len(train_active_X), "subjects.")

<class 'numpy.ndarray'>
(1099541, 13)
float64
y type: <class 'numpy.ndarray'>
y shape: (200,)
Subject S01: Epoch data shape (200, 13, 2049)
<class 'numpy.ndarray'>
(1099541, 13)
float64
y type: <class 'numpy.ndarray'>
y shape: (200,)
Subject S02: Epoch data shape (200, 13, 2049)
<class 'numpy.ndarray'>
(1105300, 13)
float64
y type: <class 'numpy.ndarray'>
y shape: (200,)
Subject S03: Epoch data shape (200, 13, 2049)
<class 'numpy.ndarray'>
(1100323, 13)
float64
y type: <class 'numpy.ndarray'>
y shape: (200,)
Subject S04: Epoch data shape (199, 13, 2049)
<class 'numpy.ndarray'>
(1105413, 13)
float64
y type: <class 'numpy.ndarray'>
y shape: (200,)
Subject S05: Epoch data shape (200, 13, 2049)
<class 'numpy.ndarray'>
(1097390, 13)
float64
y type: <class 'numpy.ndarray'>
y shape: (200,)
Subject S06: Epoch data shape (199, 13, 2049)
<class 'numpy.ndarray'>
(1096936, 13)
float64
y type: <class 'numpy.ndarray'>
y shape: (200,)
Subject S07: Epoch data shape (199, 13, 2049)
<class 'numpy.ndarra

In [5]:
 
data_dir = '/home/vishwa/eeg_tl/Recreating papers/2015_001/B'

# Lists to hold data for all subjects
eval_active_X = []         # List to hold numpy arrays with shape (n_trials, 13, n_times) per subject
eval_active_y = []         # List to hold event labels per subject
eval_active_metadata = []  # List to hold event metadata per subject

# Define subject IDs (S01 to S12 based on the dataset description)
subjects = [f'S{subj:02d}' for subj in range(1, 13)]

for subj in subjects:
    # Find all .mat files for this subject (e.g., S01_session1.mat, S01_session2.mat)
    subj_files = [f for f in os.listdir(data_dir) if f.startswith(subj) and f.endswith('.mat')]
    subj_epochs_list = []

    for filename in subj_files:
        full_filename = os.path.join(data_dir, filename)
        
        # Load the .mat file
        data_dict = loadmat(full_filename, squeeze_me=True)
        data = data_dict['data']
        X = data['X'][()]  # EEG signal (n_times, n_channels)
        print(type(X))  # Should be <class 'numpy.ndarray'>
        print(X.shape if isinstance(X, np.ndarray) else "Not a NumPy array")
        print(X.dtype if isinstance(X, np.ndarray) else "No dtype available")
        X = X.T
        y = data['y'][()]  # True labels (1 for right hand, 2 for both feet)
        print(f"y type: {type(y)}")
        if isinstance(y, np.ndarray):
            print(f"y shape: {y.shape}")
        else:
            print(f"y value: {y}")
        # print(f"Number of trials: {len(trial)}")
        trial = data['trial']  # Trial start positions in samples
        fs = data['fs']  # Sampling rate (512 Hz)

        # Define channel names and types
        ch_names = ['FC3', 'FCz', 'FC4', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP3', 'CPz', 'CP4']
        ch_types = ['eeg'] * 13
        info = mne.create_info(ch_names, sfreq=fs, ch_types=ch_types)

        # Create Raw object (transpose X to (n_channels, n_times) as required by MNE)
        raw = mne.io.RawArray(X, info, verbose=False)

        # Compute cue onset samples (cue starts 3 seconds after trial start)
        cue_samples = trial + int(3 * fs)

        # Create events array: [sample, 0, label]
        events = np.column_stack((cue_samples, np.zeros(len(cue_samples), dtype=int), y))

        # Define epoching parameters
        tmin = 0.5  
        tmax = 4.5   

        # Create epochs with baseline correction using the reference period
        epochs = mne.Epochs(
            raw,
            events,
            event_id={'1': 1, '2': 2},  # 1: right hand, 2: both feet
            tmin=tmin,
            tmax=tmax,
            baseline=None,  # Baseline from t=-3 to t=0
            preload=True,
            verbose=False
        )
        subj_epochs_list.append(epochs)

    # Concatenate epochs across sessions for this subject
    if len(subj_epochs_list) > 1:
        subj_epochs = mne.concatenate_epochs(subj_epochs_list)
    else:
        subj_epochs = subj_epochs_list[0]

    # Get the epoch data
    subj_data = subj_epochs.get_data()

    # Apply causal bandpass filter to each trial and channel
    n_trials, n_channels, n_times = subj_data.shape
    subj_filtered_data = np.empty_like(subj_data)
    for trial in range(n_trials):
        for ch in range(n_channels):
            subj_filtered_data[trial, ch, :] = causal_bandpass_filter(
                subj_data[trial, ch, :],
                lowcut=8,   # Lower bound of sensorimotor rhythm
                highcut=30, # Upper bound of sensorimotor rhythm
                fs=fs,
                order=50    # Filter order
            )

    # Append processed data, labels, and metadata
    eval_active_X.append(subj_filtered_data)
    eval_active_y.append(subj_epochs.events[:, 2])  # Labels are in the third column
    eval_active_metadata.append(subj_epochs.events)

    # Print shape to verify
    print(f"Subject {subj}: Epoch data shape {subj_filtered_data.shape}")
print("Loaded data for", len(train_active_X), "subjects.")

<class 'numpy.ndarray'>
(1097390, 13)
float64
y type: <class 'numpy.ndarray'>
y shape: (200,)
Subject S01: Epoch data shape (199, 13, 2049)
<class 'numpy.ndarray'>
(1097314, 13)
float64
y type: <class 'numpy.ndarray'>
y shape: (200,)
Subject S02: Epoch data shape (200, 13, 2049)
<class 'numpy.ndarray'>
(1096245, 13)
float64
y type: <class 'numpy.ndarray'>
y shape: (200,)
Subject S03: Epoch data shape (200, 13, 2049)
<class 'numpy.ndarray'>
(1096245, 13)
float64
y type: <class 'numpy.ndarray'>
y shape: (200,)
Subject S04: Epoch data shape (200, 13, 2049)
<class 'numpy.ndarray'>
(1097284, 13)
float64
y type: <class 'numpy.ndarray'>
y shape: (200,)
Subject S05: Epoch data shape (199, 13, 2049)
<class 'numpy.ndarray'>
(1097314, 13)
float64
y type: <class 'numpy.ndarray'>
y shape: (200,)
Subject S06: Epoch data shape (200, 13, 2049)
<class 'numpy.ndarray'>
(1096245, 13)
float64
y type: <class 'numpy.ndarray'>
y shape: (200,)
Subject S07: Epoch data shape (200, 13, 2049)
<class 'numpy.ndarra

In [66]:
train_xi = xi[:200]
test_xi = xi[200:]
train_yi = yi[:200]
test_yi = yi[200:]
print(train_xi.shape)

(200, 13, 2561)


In [67]:
n_trials = 200
n_channels = 13
train_filtered = []
eval_filtered = []
subj_filtered_data = np.empty_like(train_xi)
for trial in range(n_trials):
        for ch in range(n_channels):
            subj_filtered_data[trial, ch, :] = causal_bandpass_filter(
                train_xi[trial, ch, :],
                lowcut=8,   # Lower bound (e.g., sensorimotor rhythm)
                highcut=30, # Upper bound
                fs=512,
                order=5    # Filter order; adjust as needed for your application
            )
train_filtered.append(subj_filtered_data)

subj_filtered_data = np.empty_like(test_xi)
for trial in range(n_trials):
        for ch in range(n_channels):
            subj_filtered_data[trial, ch, :] = causal_bandpass_filter(
                test_xi[trial, ch, :],
                lowcut=8,   # Lower bound (e.g., sensorimotor rhythm)
                highcut=30, # Upper bound
                fs=512,
                order=5    # Filter order; adjust as needed for your application
            )
eval_filtered.append(subj_filtered_data)

train_filtered = np.array(train_filtered).squeeze(0)
eval_filtered = np.array(eval_filtered).squeeze(0)

In [68]:
train_filtered.shape

(200, 13, 2561)

In [70]:
pipeline = Pipeline([
            ('CSP', CSP(n_components=8, reg=None, log=True, norm_trace=False)),
            ('LDA', LinearDiscriminantAnalysis())
        ])

# Fit and predict
pipeline.fit(train_filtered, train_yi)
accuracy = pipeline.score(eval_filtered, test_yi)
print(accuracy)

Computing rank from data with rank=None
    Using tolerance 1.8e+02 (2.2e-16 eps * 13 dim * 6.4e+16  max singular value)
    Estimated rank (data): 13
    data: rank 13 computed from 13 data channels with 0 projectors
Reducing data rank from 13 -> 13
Estimating class=feet covariance using EMPIRICAL
Done.
Estimating class=right_hand covariance using EMPIRICAL
Done.
0.98
